In [1]:
import pandas as pd
import yfinance as yf
import time
import os
import re
from IPython.display import clear_output

In [2]:
#ADJ is the amount in dollars to make up for the price diff bw/ yfinance api and broker data
yfinance_sym_dic = { 
    'MNQ': {'SYM':'MNQ=F', 'ADJ': 0},
    'NQ': {'SYM':'NQ=F', 'ADJ': 0},
    'US100': {'SYM':'MNQ=F', 'ADJ': -53.18},
    'GC': {'SYM':'GC=F', 'ADJ': 0},
    'MGX': {'SYM':'MGC=F', 'ADJ': 0},
    'SI': {'SYM':'SI=F', 'ADJ': 0},
    'SIL': {'SYM':'SIL=F', 'ADJ': 0},
    'XAUUSD': {'SYM':'GC=F', 'ADJ': 0},
    'AGXUSD': {'SYM':'SI=F', 'ADJ': 0},
    'BZ': {'SYM':'BZ=F', 'ADJ': 0}, # Brent Crude Futures
    'CL': {'SYM':'CL=F', 'ADJ': 0}, # WTI Crude Futures
    'BTC': {'SYM':'BTC-USD', 'ADJ': 0},
    'ETH': {'SYM':'ETH-USD', 'ADJ': 0}
}


def get_live_price(ticker_symbol: str, yfinance_map: dict)-> float:
    # Initialize the Ticker object 
    if ticker_symbol in yfinance_map.keys():
        ticker = yf.Ticker(yfinance_map[ticker_symbol]['SYM'])
        # .fast_info provides the most recent 'last_price'
        # This is faster than fetching the full .info dictionary
        current_price = ticker.fast_info['last_price'] + yfinance_map[ticker_symbol]['ADJ']
    else:
        ticker = yf.Ticker(ticker_symbol)
        current_price = ticker.fast_info['last_price']
        
    return current_price

# Read the trades worksheet

In [3]:
# 1. Replace with your actual Google Sheet ID
# (Found in the URL: https://docs.google.com/spreadsheets/d/SHEET_ID/edit)
SHEET_ID = "1HJ9h7UEtUQCXNA58UkZyPsHogJWBAcB1lNWt9nOPMR4"

# 2. Specify the tab name (optional, defaults to the first sheet)
SHEET_NAME = 'Trades'

# 3. Construct the export URL
url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME}"

# 4. Load into DataFrame
df = pd.read_csv(url)

# 5. Keep open trades only
df = df[df.Closed=='No'].copy()

# Fill numberic columns' NA with 0 and cast numeric columns from str to float type
cols = ['Open Price', 'Close Price', 'Commission','Risk ($)', 'Balance at Open', 'PnL']
df[cols] = df[cols].fillna('0')
for c in cols:
    df[c] = df[c].apply(lambda x: float(re.sub(r"\(", "-", re.sub(r"[,\)]", "", x))))
df.head()

,Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Risk ($),PnL,Closed,Close Date,Entry Link,Exit Link 1,Exit Link 2
29,8/24/2026,Paper Trading #1,COF,-34.0,221.02,217.37,0.00,59800.00,-97.92,124.10,No,NaN,Link,NaN,NaN
32,8/24/2026,Tradestation - Equity,COF,-34.0,220.62,217.37,0.00,42336.39,-115.94,110.50,No,NaN,Link,NaN,NaN
33,8/24/2026,Tradestation - Equity,CVNA,-130.0,72.08,81.08,-2.91,42336.39,-1172.61,-1172.61,No,NaN,Link,NaN,NaN
36,8/25/2026,Tradestation - Futures,MNQ,-4.0,29292.00,29480.75,0.00,13015.34,0.00,-1510.00,No,NaN,Link,NaN,NaN
40,8/25/2026,Paper Trading #1,CVNA,-171.0,75.57,73.95,0.00,59313.65,-1185.03,277.02,No,NaN,NaN,NaN,NaN


# Get Point Values

In [4]:
# 2. Specify the tab name (optional, defaults to the first sheet)
SHEET_NAME = 'Symbols'

# 3. Construct the export URL
url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME}"

# 4. Load into DataFrame
point_val_df = pd.read_csv(url, header=None, names=['Symbol', 'Point Value'])
point_val_df.head()

,Symbol,Point Value
0,COF,1
1,CVNA,1
2,MNQ,2
3,QQQ,1
4,UKOIL,1


# Get Prices

In [5]:
price_df = pd.DataFrame(df['Symbol']).drop_duplicates()
price_df['Current Price'] = price_df.Symbol.apply(lambda x : get_live_price(x, yfinance_sym_dic))
price_df

,Symbol,Current Price
29,COF,217.729996
33,CVNA,73.839996
36,MNQ,29323.750000
61,US100,29270.570000


# Append Price to trades DF

In [6]:
df = pd.merge(df, price_df, on='Symbol', how='left')
df = pd.merge(df, point_val_df, on='Symbol', how='left')
df['Point Value'] = df['Point Value'].fillna(1)
df['PnL'] = (df['Volume'] * (df['Current Price']-df['Open Price']) * df['Point Value']).round(2)
df

,Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Risk ($),PnL,Closed,Close Date,Entry Link,Exit Link 1,Exit Link 2,Current Price,Point Value
0,8/24/2026,Paper Trading #1,COF,-34.0,221.02,217.37,0.00,59800.00,-97.92,111.86,No,NaN,Link,NaN,NaN,217.729996,1
1,8/24/2026,Tradestation - Equity,COF,-34.0,220.62,217.37,0.00,42336.39,-115.94,98.26,No,NaN,Link,NaN,NaN,217.729996,1
2,8/24/2026,Tradestation - Equity,CVNA,-130.0,72.08,81.08,-2.91,42336.39,-1172.61,-228.80,No,NaN,Link,NaN,NaN,73.839996,1
3,8/25/2026,Tradestation - Futures,MNQ,-4.0,29292.00,29480.75,0.00,13015.34,0.00,-254.00,No,NaN,Link,NaN,NaN,29323.750000,2
4,8/25/2026,Paper Trading #1,CVNA,-171.0,75.57,73.95,0.00,59313.65,-1185.03,295.83,No,NaN,NaN,NaN,NaN,73.839996,1
5,8/26/2026,FTP - 147759,US100,20.0,29268.70,29263.45,0.00,98116.20,-105.00,37.40,No,NaN,NaN,NaN,NaN,29270.570000,1


# Group by account and symbol to report

In [8]:
out = df.groupby(['Symbol','Account']).agg({'Volume': sum, 'PnL': sum})
print(out)

                               Volume     PnL
Symbol Account                               
COF    Paper Trading #1         -34.0  111.86
       Tradestation - Equity    -34.0   98.26
CVNA   Paper Trading #1        -171.0  295.83
       Tradestation - Equity   -130.0 -228.80
MNQ    Tradestation - Futures    -4.0 -254.00
US100  FTP - 147759              20.0   37.40
